In [ ]:
import csv
import mysql.connector
from datetime import datetime

DB_CONFIG = {
    "host": "localhost",
    "user": "root",
    "password": "YOUR_PASSWORD",
    "database": "lowes_db",
}

CSV_PATH = "data/raw/lowes_financials_complete.csv"


def parse_date(val):
    if not val:
        return None
    for fmt in ("%m/%d/%y", "%m/%d/%Y"):
        try:
            return datetime.strptime(val, fmt).strftime("%Y-%m-%d")
        except ValueError:
            continue
    return None


def to_null(val):
    return None if val == "" else val


INSERT_SQL = """
    INSERT INTO financials (
        fiscal_year, fiscal_quarter, filing_type, period_end_date,
        net_sales_mm, cost_of_goods_mm, gross_margin_mm, operating_income_mm,
        net_income_mm, diluted_earn_per_share, compare_sales_pct,
        store_count, selling_sqft_mm, capex_mm, inventory_mm,
        employee_count, source_url
    ) VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
    ON DUPLICATE KEY UPDATE
        filing_type            = VALUES(filing_type),
        period_end_date        = VALUES(period_end_date),
        net_sales_mm           = VALUES(net_sales_mm),
        cost_of_goods_mm       = VALUES(cost_of_goods_mm),
        gross_margin_mm        = VALUES(gross_margin_mm),
        operating_income_mm    = VALUES(operating_income_mm),
        net_income_mm          = VALUES(net_income_mm),
        diluted_earn_per_share = VALUES(diluted_earn_per_share),
        compare_sales_pct      = VALUES(compare_sales_pct),
        store_count            = VALUES(store_count),
        selling_sqft_mm        = VALUES(selling_sqft_mm),
        capex_mm               = VALUES(capex_mm),
        inventory_mm           = VALUES(inventory_mm),
        employee_count         = VALUES(employee_count),
        source_url             = VALUES(source_url)
"""

def main():
    conn = mysql.connector.connect(**DB_CONFIG)
    cursor = conn.cursor()

    inserted = updated = skipped = 0

    with open(CSV_PATH, newline="", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for row in reader:
            if not row["fiscal_year"] or not row["fiscal_quarter"]:
                skipped += 1
                continue
            params = (
                int(row["fiscal_year"]),
                row["fiscal_quarter"],
                row["filing_type"],
                parse_date(row["period_end_date"]),
                to_null(row["net_sales_mm"]),
                to_null(row["cost_of_goods_mm"]),
                to_null(row["gross_margin_mm"]),
                to_null(row["operating_income_mm"]),
                to_null(row["net_income_mm"]),
                to_null(row["diluted_earn_per_share"]),
                to_null(row["compare_sales_pct"]),
                to_null(row["store_count"]),
                to_null(row["selling_sqft_mm"]),
                to_null(row["capex_mm"]),
                to_null(row["inventory_mm"]),
                to_null(row["employee_count"]),
                to_null(row["source_url"]),
            )
            cursor.execute(INSERT_SQL, params)
            if cursor.rowcount == 1:
                inserted += 1
            elif cursor.rowcount == 2:
                updated += 1

    conn.commit()
    cursor.close()
    conn.close()

main()


Table ready.
Done — inserted: 15, updated: 0, skipped: 2
